In [1]:
from src.configs.pkpd_registry import ACTIVE_MODEL, GLOBAL_SEED, n_subjects
from src.data.simulate_pkpd import generate_population_data

pop_data, subject_params, active_cfg = generate_population_data(
    model_name=ACTIVE_MODEL,
    seed=GLOBAL_SEED,
    n_subjects=n_subjects
)

print(pop_data.shape, active_cfg["family"], len(subject_params))
print(pop_data[:3])

(144, 4) idr 12
[[ 0.          0.          0.         34.88986279]
 [ 0.          0.5         3.74906565 32.4370152 ]
 [ 0.          1.          5.60952484 26.81141074]]


In [2]:
import torch
from deepymod import DeepMoD
from src.models.network import TimeOnlyNet
from src.models.library import PDLibraryExpanded
from src.models.estimator import FixedMaskEstimator
from src.models.constraint import RidgeConstraint

device = "cuda" if torch.cuda.is_available() else "cpu"

n_terms = len(PDLibraryExpanded.term_names())
init_mask = torch.ones(n_terms, dtype=torch.bool, device=device)

network = TimeOnlyNet().to(device)
library = PDLibraryExpanded().to(device)
estimator = FixedMaskEstimator(init_mask).to(device)
constraint = RidgeConstraint(lam=1e-4).to(device)

model = DeepMoD(network, library, estimator, constraint).to(device)
print("Model init OK, terms =", PDLibraryExpanded.term_names())

Model init OK, terms = ['1', 'R', 'C', 'C^2', 'Emax(C)', 'Hill(C)', 'C*R', 'Emax(C)*R', 'Hill(C)*R']


In [3]:
from src.training.pruning import prune_mask_general
from src.training.trainer import build_train_loader, run_train_loop, get_coeff_and_mask
from src.training.ranking import rank_topk_validation_bic
print("Step4 modules import OK")

Step4 modules import OK


In [4]:
from src.configs.defaults import DEFAULTS
from src.configs.pkpd_registry import ACTIVE_MODEL, GLOBAL_SEED, n_subjects
from src.data.simulate_pkpd import generate_population_data
from src.pipeline.run_single import run_single_discovery

pop_data, subject_params, cfg = generate_population_data(
    model_name=ACTIVE_MODEL,
    seed=GLOBAL_SEED,
    n_subjects=n_subjects
)

res = run_single_discovery(
    pop_data=pop_data,
    active_model=ACTIVE_MODEL,
    config=DEFAULTS,
)

Preprocessing data
Dataset is using device:  cuda

[Warmup]
warmup done | loss=0.002985 mse=0.002736 reg=0.000194

[Iterative pruning]
Round 1:
  active(old): [np.str_('1'), np.str_('R'), np.str_('C'), np.str_('C^2'), np.str_('Emax(C)'), np.str_('Hill(C)'), np.str_('C*R'), np.str_('Emax(C)*R'), np.str_('Hill(C)*R')]
  active(new): [np.str_('1'), np.str_('R'), np.str_('Hill(C)')]
  retrain done | loss=0.000441 mse=0.000220 reg=0.000042
Round 2:
  active(old): [np.str_('1'), np.str_('R'), np.str_('Hill(C)')]
  active(new): [np.str_('1'), np.str_('R'), np.str_('Hill(C)')]
  -> unchanged (1/2)
Round 3:
  active(old): [np.str_('1'), np.str_('R'), np.str_('Hill(C)')]
  active(new): [np.str_('1'), np.str_('R'), np.str_('Hill(C)')]
  -> unchanged (2/2)
  -> mask stable enough, stop.

[Top-K candidate generation with Validation-BIC]

=== Top-K Candidate Structures (Validation BIC) ===
[Rank 1] BIC_val=0.546539 | mse_val=0.405317 | mse_train=0.000101 | k=3
  terms: ['1', 'R', 'Hill(C)']
[Rank 2]

In [5]:
from src.pipeline.export_for_nlme import export_nlme_inputs

data_csv, cand_json = export_nlme_inputs(
    pop_data=pop_data,                 # 你前面生成的pop_data
    top_results=res["top_results"],    # Step5输出
    out_dir="../artifacts/nlme"
)
print(data_csv)
print(cand_json)

../artifacts/nlme/pkpd_long.csv
../artifacts/nlme/topk_candidates.json


In [6]:
from src.pipeline.run_nlme_validation import run_nlme_validation

df_nlme = run_nlme_validation(
    pop_data=pop_data,
    top_results=res["top_results"],
    project_root=".."
)
display(df_nlme)

,rank,terms,converged,logLik,AIC,BIC,MSE,message
0,1,1 + R + Hill(C),0,NaN,NaN,NaN,NaN,MATLAB:invalidConversion | 无法从 cell 转换为 double。
1,2,1 + R + Emax(C) + Hill(C),0,NaN,NaN,NaN,NaN,MATLAB:invalidConversion | 无法从 cell 转换为 double。
2,3,1 + R + Hill(C) + Hill(C)*R,0,NaN,NaN,NaN,NaN,MATLAB:invalidConversion | 无法从 cell 转换为 double。
3,4,1 + R + C + Hill(C),0,NaN,NaN,NaN,NaN,MATLAB:invalidConversion | 无法从 cell 转换为 double。
4,5,1 + R + C^2 + Hill(C),0,NaN,NaN,NaN,NaN,MATLAB:invalidConversion | 无法从 cell 转换为 double。
